In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits

print("Libraries Loaded Successfully ✅")

Libraries Loaded Successfully ✅


In [ ]:
import os

ROOT = "/content/drive/MyDrive/Exoplanet_Dataset/03_Light_Curves"

first_fits = None

for root, dirs, files in os.walk(ROOT):
    for file in files:
        if file.endswith(".fits"):
            first_fits = os.path.join(root, file)
            break
    if first_fits:
        break

print(first_fits)

/content/drive/MyDrive/Exoplanet_Dataset/03_Light_Curves/TIC_58466459/mastDownload/TESS/tess2019085135100-s0010-0000000058466459-0140-s/tess2019085135100-s0010-0000000058466459-0140-s_lc.fits


In [ ]:
from astropy.io import fits

hdul = fits.open(first_fits)

hdul.info()

Filename: /content/drive/MyDrive/Exoplanet_Dataset/03_Light_Curves/TIC_58466459/mastDownload/TESS/tess2019085135100-s0010-0000000058466459-0140-s/tess2019085135100-s0010-0000000058466459-0140-s_lc.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      44   ()      
  1  LIGHTCURVE    1 BinTableHDU    167   18900R x 20C   [D, E, J, E, E, E, E, E, E, J, D, E, D, E, D, E, D, E, E, E]   
  2  APERTURE      1 ImageHDU        49   (11, 11)   int32   


In [ ]:
lightcurve = hdul[1].data

print(lightcurve.columns)

ColDefs(
    name = 'TIME'; format = 'D'; unit = 'BJD - 2457000, days'; disp = 'D14.7'
    name = 'TIMECORR'; format = 'E'; unit = 'd'; disp = 'E14.7'
    name = 'CADENCENO'; format = 'J'; disp = 'I10'
    name = 'SAP_FLUX'; format = 'E'; unit = 'e-/s'; disp = 'E14.7'
    name = 'SAP_FLUX_ERR'; format = 'E'; unit = 'e-/s'; disp = 'E14.7'
    name = 'SAP_BKG'; format = 'E'; unit = 'e-/s'; disp = 'E14.7'
    name = 'SAP_BKG_ERR'; format = 'E'; unit = 'e-/s'; disp = 'E14.7'
    name = 'PDCSAP_FLUX'; format = 'E'; unit = 'e-/s'; disp = 'E14.7'
    name = 'PDCSAP_FLUX_ERR'; format = 'E'; unit = 'e-/s'; disp = 'E14.7'
    name = 'QUALITY'; format = 'J'; disp = 'B16.16'
    name = 'PSF_CENTR1'; format = 'D'; unit = 'pixel'; disp = 'F10.5'
    name = 'PSF_CENTR1_ERR'; format = 'E'; unit = 'pixel'; disp = 'E14.7'
    name = 'PSF_CENTR2'; format = 'D'; unit = 'pixel'; disp = 'F10.5'
    name = 'PSF_CENTR2_ERR'; format = 'E'; unit = 'pixel'; disp = 'E14.7'
    name = 'MOM_CENTR1'; format = 'D'; u

In [ ]:
for i, col in enumerate(lightcurve.columns.names):
    print(f"{i:02d} : {col}")

00 : TIME
01 : TIMECORR
02 : CADENCENO
03 : SAP_FLUX
04 : SAP_FLUX_ERR
05 : SAP_BKG
06 : SAP_BKG_ERR
07 : PDCSAP_FLUX
08 : PDCSAP_FLUX_ERR
09 : QUALITY
10 : PSF_CENTR1
11 : PSF_CENTR1_ERR
12 : PSF_CENTR2
13 : PSF_CENTR2_ERR
14 : MOM_CENTR1
15 : MOM_CENTR1_ERR
16 : MOM_CENTR2
17 : MOM_CENTR2_ERR
18 : POS_CORR1
19 : POS_CORR2


In [ ]:
df = pd.DataFrame(lightcurve)

print(df.head())

          TIME  TIMECORR  CADENCENO     SAP_FLUX  SAP_FLUX_ERR       SAP_BKG  \
0  1569.436731  0.005002     246223          NaN           NaN           NaN   
1  1569.438120  0.005002     246224          NaN           NaN           NaN   
2  1569.439508  0.005002     246225          NaN           NaN           NaN   
3  1569.440897  0.005002     246226          NaN           NaN           NaN   
4  1569.442286  0.005002     246227  9166.600586       28.7946  38469.945312   

   SAP_BKG_ERR  PDCSAP_FLUX  PDCSAP_FLUX_ERR  QUALITY  PSF_CENTR1  \
0          NaN          NaN              NaN     8360         NaN   
1          NaN          NaN              NaN     8224         NaN   
2          NaN          NaN              NaN     8224         NaN   
3          NaN          NaN              NaN     8224         NaN   
4    16.851168          NaN              NaN    12288         NaN   

   PSF_CENTR1_ERR  PSF_CENTR2  PSF_CENTR2_ERR  MOM_CENTR1  MOM_CENTR1_ERR  \
0             NaN         N

In [ ]:
# ==========================================================
# STEP 2 : Convert ONE FITS to Structured Light Curve Table
# ==========================================================

import pandas as pd
import re
from astropy.io import fits

# Open FITS
hdul = fits.open(first_fits)

# Read Light Curve Extension
lc = hdul[1].data

# Convert to DataFrame
df = pd.DataFrame(lc)

# ----------------------------------------------------------
# Extract TIC ID
# ----------------------------------------------------------

tic = hdul[0].header["TICID"]

# ----------------------------------------------------------
# Extract Sector
# ----------------------------------------------------------

match = re.search(r"s(\d{4})", first_fits)

sector = int(match.group(1)) if match else -1

# ----------------------------------------------------------
# Get Label
# ----------------------------------------------------------

label = prototype.loc[
    prototype["TIC_ID"] == tic,
    "Label"
].values[0]

# ----------------------------------------------------------
# Keep Only Required Columns
# ----------------------------------------------------------

structured = pd.DataFrame({

    "TIC_ID": tic,

    "Label": label,

    "Sector": sector,

    "Time": df["TIME"],

    "Flux": df["PDCSAP_FLUX"],

    "Flux_Error": df["PDCSAP_FLUX_ERR"],

    "Quality": df["QUALITY"],

    "Cadence": df["CADENCENO"]

})

structured.head()

,TIC_ID,Label,Sector,Time,Flux,Flux_Error,Quality,Cadence
0,58466459,EB,2019,1569.436731,NaN,NaN,8360,246223
1,58466459,EB,2019,1569.438120,NaN,NaN,8224,246224
2,58466459,EB,2019,1569.439508,NaN,NaN,8224,246225
3,58466459,EB,2019,1569.440897,NaN,NaN,8224,246226
4,58466459,EB,2019,1569.442286,NaN,NaN,12288,246227


In [ ]:
print(structured.shape)

structured.info()

structured.head(10)

(18900, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18900 entries, 0 to 18899
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   TIC_ID      18900 non-null  int64  
 1   Label       18900 non-null  object 
 2   Sector      18900 non-null  int64  
 3   Time        18197 non-null  float64
 4   Flux        14945 non-null  float32
 5   Flux_Error  14945 non-null  float32
 6   Quality     18900 non-null  int32  
 7   Cadence     18900 non-null  int32  
dtypes: float32(2), float64(1), int32(2), int64(2), object(1)
memory usage: 886.1+ KB


,TIC_ID,Label,Sector,Time,Flux,Flux_Error,Quality,Cadence
0,58466459,EB,2019,1569.436731,NaN,NaN,8360,246223
1,58466459,EB,2019,1569.438120,NaN,NaN,8224,246224
2,58466459,EB,2019,1569.439508,NaN,NaN,8224,246225
3,58466459,EB,2019,1569.440897,NaN,NaN,8224,246226
4,58466459,EB,2019,1569.442286,NaN,NaN,12288,246227
5,58466459,EB,2019,1569.443675,NaN,NaN,12288,246228
6,58466459,EB,2019,1569.445064,NaN,NaN,12288,246229
7,58466459,EB,2019,1569.446453,NaN,NaN,12288,246230
8,58466459,EB,2019,1569.447842,NaN,NaN,12288,246231
9,58466459,EB,2019,1569.449231,NaN,NaN,12288,246232


In [ ]:
# ==========================================================
# FITS → Structured Light Curve Tables
# One CSV per TIC
# ==========================================================

import os
import re
import pandas as pd
from astropy.io import fits
from tqdm import tqdm

INPUT_ROOT = "/content/drive/MyDrive/Exoplanet_Dataset/03_Light_Curves"

OUTPUT_ROOT = "/content/drive/MyDrive/Exoplanet_Dataset/04_Structured_LightCurves"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

prototype = pd.read_csv(
    "/content/drive/MyDrive/Exoplanet_Dataset/02_Master_Index/master_labels_prototype.csv"
)

label_dict = dict(zip(prototype["TIC_ID"], prototype["Label"]))

star_folders = sorted(os.listdir(INPUT_ROOT))

converted = 0

for folder in tqdm(star_folders):

    if not folder.startswith("TIC_"):
        continue

    tic = int(folder.replace("TIC_", ""))

    label = label_dict.get(tic, "Unknown")

    all_sectors = []

    folder_path = os.path.join(INPUT_ROOT, folder)

    for root, dirs, files in os.walk(folder_path):

        for file in files:

            if not file.endswith(".fits"):
                continue

            fits_path = os.path.join(root, file)

            try:

                hdul = fits.open(fits_path)

                df = pd.DataFrame(hdul[1].data)

                match = re.search(r"-s(\d{4})-", file)

                sector = int(match.group(1)) if match else -1

                structured = pd.DataFrame({

                    "TIC_ID": tic,

                    "Label": label,

                    "Sector": sector,

                    "Time": df["TIME"],

                    "Flux": df["PDCSAP_FLUX"],

                    "Flux_Error": df["PDCSAP_FLUX_ERR"],

                    "Quality": df["QUALITY"],

                    "Cadence": df["CADENCENO"]

                })

                all_sectors.append(structured)

                hdul.close()

            except Exception:
                continue

    if len(all_sectors) == 0:
        continue

    final_df = pd.concat(all_sectors, ignore_index=True)

    final_df.to_csv(

        os.path.join(
            OUTPUT_ROOT,
            f"TIC_{tic}.csv"
        ),

        index=False

    )

    converted += 1

print("="*60)
print("Conversion Finished")
print("="*60)

print("Stars Converted :", converted)

100%|██████████| 1200/1200 [2:53:00<00:00,  8.65s/it]

Conversion Finished
Stars Converted : 1172


In [ ]:
import os

OUTPUT = "/content/drive/MyDrive/Exoplanet_Dataset/04_Structured_LightCurves"

print("CSV Files :", len(os.listdir(OUTPUT)))

CSV Files : 1172


In [ ]:
import pandas as pd
import os

OUTPUT = "/content/drive/MyDrive/Exoplanet_Dataset/04_Structured_LightCurves"

sample = os.listdir(OUTPUT)[0]

df = pd.read_csv(os.path.join(OUTPUT, sample))

print(df.shape)
print(df.head())

(57998, 8)
      TIC_ID Label  Sector         Time       Flux  Flux_Error  Quality  \
0  100100829    EB       2  1354.105953        NaN         NaN      168   
1  100100829    EB       2  1354.107342        NaN         NaN      160   
2  100100829    EB       2  1354.108731        NaN         NaN       32   
3  100100829    EB       2  1354.110120        NaN         NaN       32   
4  100100829    EB       2  1354.111509  1678.6504    9.462868        0   

   Cadence  
0    91186  
1    91187  
2    91188  
3    91189  
4    91190  
